# INF-473 - Introducción a la Inteligencia Artificial Explicable
## 1er Semestre 2026 -- Prof. Raquel Pezoa
<center>
     <h1>Inferencia Variacional (VI)</h1>
     <h2>Regresión Lineal<h2>
</center>


## Idea de esta versión


Este notebook está basado en: https://github.com/tensorchiefs/dl_book/blob/master/chapter_08/nb_ch08_01.ipynb, pero no usamos `tensorflow_probability` sino que solo usa `TensorFlow/Keras`.





## Bibliotecas

In [ ]:
%matplotlib inline

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

## Datos

Usaremos datos sintéticos generados desde

$$
y = 2x - 1 + \varepsilon,
\qquad
\varepsilon \sim \mathcal{N}(0, \sigma^2).
$$

El término  $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ representa el ruido aleatorio o **incertidumbre de los datos**.

In [ ]:
np.random.seed(2)
tf.random.set_seed(2)

n = 4 #solo 4 puntos!
sigma = 1.0

x_np = np.linspace(-2, 2, n).reshape((n, 1)).astype("float32")
y_np = (2 * x_np[:, 0] - 1 + np.random.normal(0, sigma, n)).astype("float32")

x = tf.constant(x_np)
y = tf.constant(y_np.reshape(-1, 1))

In [ ]:
plt.scatter(x_np, y_np, c="red")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Datos sintéticos")
plt.show()

## Modelo probabilístico

* Queremos modelar $y$ que es  aleatoria por el ruido de los datos, y en este ejemplo tenemos varianza constante.

$$
y_i \mid x_i, a, b \sim \mathcal{N}(a x_i + b, \sigma^2),
$$

* y los valores de para $a$ y $b$ no son puntuales, sino que distribuciones.

* En inferencia variacional (VI) aproximamos la posterior de los parámetros $a$ y $b$ mediante:

$$
q_\lambda(a) = \mathcal{N}(\mu_a, \sigma_a^2),
\qquad
q_\lambda(b) = \mathcal{N}(\mu_b, \sigma_b^2).
$$

* Entonces los parámetros entrenables son:

$$
\lambda = (\mu_a, \rho_a, \mu_b, \rho_b),
$$

* donde usamos

$$
\sigma_a = \operatorname{softplus}(\rho_a),
\qquad
\sigma_b = \operatorname{softplus}(\rho_b),
$$

* para asegurar que las desviaciones estándar sean positivas.

## Distribució aproximada

* La idea principal es aprender una distribución para los parámetros del modelo.

* Usaremos un modelo lineal, pero ahora se busca **aprender una distribución** para los parámetros $a$ y $b$ de la regresión.

* Para cada parámetro $a$ y $b$, se supone una **distribución Normal** con parámetros $\mu$ y $\sigma$. Por lo tanto, se tiene que aprender $\mu_a$, $\sigma_a$, $\mu_b$ y $\sigma_b$.


* Como vimos en clases, al usar inferencia variacional (VI). no trabajamos directamente con la verdadera distribución posterior $p(\theta\mid D)$, ya que es muy difícil de obtener.

* Usamos una distribución aproximada, que denotamos $q_\lambda(\theta)$, donde $\theta$ representa los parámetros del modelo y $\lambda$ corresponde a los parámetros de la distribución aproximada.

* ¿Qué se espera? que $p(\theta\mid D)$ sea muy parecida con $q_\lambda(\theta) \rightarrow$ usar ``Kullback-Leibler (KL)``.

## Función de pérdida con VI

* La **función de pérdida** es:

$$
\text{loss}_{\text{VI}} = \text{KL}[q_{\lambda}(\theta) || p(\theta)] - E_{\theta \sim q_{\lambda}} [\log{p(D|\theta)}] = \text{loss}_{\text{KL}} + \text{loss}_{\text{NLL}}
$$

y vemos que tiene dos términos principales.

* ``El primer término``: 

$$\text{loss}_{\text{KL}} = \text{KL}[q_{\lambda}(\theta) || p(\theta)]$$

mide qué tan distinta es la distribución aproximada $q_{\lambda}(\theta)$ respecto al prior $p(\theta)$.

* Si suponemos que el  prior $p(\theta) \sim \mathcal{N}(0,1)$, entones la KL tiene solución analítica:


$$\text{loss}_{\text{KL}}=  \text{KL}[q_{\lambda}(\theta)||p(\theta)]  = \text{KL}[N(\mu,\sigma)||N(0,1)] = -\frac{1}{2}(1 + \log{(\sigma^2)} - \mu^2 -\sigma^2 )$$

lo cual es conveniente ya que no necesitamos (en este caso) aproximarla numéricamente.


* ``El segundo término``:

$$\text{loss}_{\text{NLL}} = - E_{\theta \sim q_{\lambda}} [\log{p(D|\theta)}]$$

busca explicar bien los datos.

* Ahora los parámetros $\theta$ son aleatorios, y por lo tanto la likelihood también es aleatoria.

* Debemos calcular un valor esperado sobre muchos posibles parámetros.

## ¿Por qué muestreamos?

* Como $\theta \sim q_{\lambda}(\theta)$ los parámetros cambian constantemente.
* Entonces el procedimiento conceptual es:
1. Muestrear parámetros: $\theta^{(1)}$
2. Calcular: $\log{p(D \mid  \theta^{(1)})}$
3. Promediar sobre varios muestreos.


* En la práctica. ¿Cuántos muestreos se necesitan? **Solo uno**.
  * Podríamos hacer muchos muestreos pero,
  * usando gradient descent estocástico
  * minibatches
  * muchas iteraciones de entrenamiento $\rightarrow$ con un solo sample por batch es suficiente para obtener una buena aproximación.

* Lo importante es estimar bien el gradiente:

  $$\nabla_{\lambda} \, \mathbb{E}_{\theta \sim q_{\lambda}} \left[ \log p(D \mid \theta) \right]$$

es decir, queremos saber hacia donde mover $\mu$ y $\sigma$ para mejorar el modelo.

* Si bien una sola muestra produce un gradiente ruidoso, pero el ruido no tiene sesgo, y al repetir miles de actualizaciones el promedio de esos gradientes finalmente apunta en la dirección correcta.

* estamos tomando una muestra, pero en cada iteración del entrenamiento.

* En cada batch se usa un $a$ y $b$ distintos.

## Otro problema

* Sabemos que podemos muestrear:

$$\theta \sim q_{\lambda}$$

pero $\theta$ depende de $\mu$ y $\sigma$

* Muestrear no es una operación diferenciable.pero n

* ¿Cómo derivamos respecto a $\mu$ y $\sigma$ si hay un muestreo aleatorio involucrado? $\rightarrow$ **truco de reparametrización**.

* Se escribe el muestreo aleatorio de la siguiente forma:

$$ \theta = \mu + \sigma \varepsilon$$

* En particular,  en vez de muestrear $a\sim N(\mu_a, \sigma_a)$ se calcula $a_{\text{rep}} = \mu_a + \sigma_a \cdot \epsilon $ y se muestrea $\epsilon \sim N(0,1)$ y se puede hacer backpropagation con respecto a $\sigma_a$ y $\mu_a$

* Finalmente, se quiere minimizar la esperanza de la pérdida (como la log-verosimilitud negativa), y ahora podemos estimar esa esperanza con Monte Carlo (una muestra basta, como ya vimos), y calcular derivadas respecto a los parámetros de la distribución.

## Código de la función de pérdida

* ¿De dónde viene esa expresión?

In [ ]:
def normal_log_prob(y, loc, scale):
    """Log-probabilidad de una Normal N(loc, scale^2)."""
    scale = tf.cast(scale, y.dtype)
    return (
        -0.5 * tf.math.log(tf.constant(2.0 * np.pi, dtype=y.dtype))
        - tf.math.log(scale)
        - 0.5 * tf.square((y - loc) / scale)
    )


def kl_normal_q_p(mu_q, sigma_q, mu_p=0.0, sigma_p=1.0):
    """
    KL[q || p] para q=N(mu_q, sigma_q^2) y p=N(mu_p, sigma_p^2).
    En el caso estándar p=N(0,1), queda:
    0.5 * (sigma_q^2 + mu_q^2 - 1 - log(sigma_q^2)).
    """
    mu_p = tf.cast(mu_p, mu_q.dtype)
    sigma_p = tf.cast(sigma_p, mu_q.dtype)

    return (
        tf.math.log(sigma_p / sigma_q)
        + (tf.square(sigma_q) + tf.square(mu_q - mu_p)) / (2.0 * tf.square(sigma_p))
        - 0.5
    )

## Entrenamiento de regresión lineal variacional

Entonces, en cada época:

1. se calcula $\sigma_a$ y $\sigma_b$ usando `softplus`;
2. se toma una muestra usando reparametrización:

$$
a = \mu_a + \sigma_a \varepsilon_a,
\qquad
b = \mu_b + \sigma_b \varepsilon_b,
\qquad
\varepsilon_a, \varepsilon_b \sim \mathcal{N}(0,1);
$$

3. se evalúa la log-verosimilitud de los datos bajo esa muestra;
4. se suma el término KL;
5. se actualizan los parámetros variacionales.

In [ ]:
class Logger:
    def __init__(self, steps, num_weights=4):
        self.steps = steps
        self.num_weights = num_weights
        self.X = np.zeros((steps, 12))
        self.header = "epoch,mu_a,rho_a,mu_b,rho_b,g_mu_a,g_rho_a,g_mu_b,g_rho_b,loss,loss_kl,loss_nll"

    def log(self, step, epoch, w, w_grad, loss, loss_kl, loss_nll):
        n = self.num_weights
        self.X[step, 0] = epoch
        self.X[step, 1:(n+1)] = w.numpy()
        self.X[step, (n+1):((2*n)+1)] = w_grad.numpy()
        self.X[step, ((2*n)+1)] = loss.numpy()
        self.X[step, ((2*n)+2)] = loss_kl.numpy()
        self.X[step, ((2*n)+3)] = loss_nll.numpy()

In [ ]:
epochs = 10000
learning_rate = 0.001

optimizer = tf.keras.optimizers.SGD(learning_rate)
logger = Logger(epochs)

# Parámetros iniciales: mu_a, rho_a, mu_b, rho_b.
w = tf.Variable([1.0, 1.0, 1.0, 1.0], dtype=tf.float32)

for epoch in range(epochs):
    with tf.GradientTape() as tape:
        mu_a = w[0]
        sigma_a = tf.nn.softplus(w[1])

        mu_b = w[2]
        sigma_b = tf.nn.softplus(w[3])

        # KL[q(a)||p(a)] + KL[q(b)||p(b)], con prior estándar N(0,1).
        loss_kl = (
            kl_normal_q_p(mu_a, sigma_a)
            + kl_normal_q_p(mu_b, sigma_b)
        )

        # Reparametrización.
        eps_a = tf.random.normal(shape=())
        eps_b = tf.random.normal(shape=())

        sample_a = mu_a + sigma_a * eps_a
        sample_b = mu_b + sigma_b * eps_b

        # Media predictiva para esta muestra de parámetros.
        y_loc = sample_a * x + sample_b

        # Negative log-likelihood.
        log_likelihood = tf.reduce_sum(normal_log_prob(y, loc=y_loc, scale=sigma))
        loss_nll = -log_likelihood

        # ELBO negativa: NLL + KL.
        total_loss = loss_nll + loss_kl

    gradients = tape.gradient(total_loss, w)
    optimizer.apply_gradients([(gradients, w)])

    logger.log(epoch, epoch, w, gradients, total_loss, loss_kl, loss_nll)

    if (epoch + 1) % 2000 == 0 or epoch == 0:
        print(f"Época {epoch + 1}: pérdida = {total_loss.numpy():.4f}")
        print(f"mu_a = {w[0].numpy():.4f}, mu_b = {w[2].numpy():.4f}")

## Parámetros aprendidos

Aquí **no** se debe aplicar `softplus` a `mu_a` ni a `mu_b`.
`softplus` solo se aplica a los parámetros que representan desviaciones estándar.

In [ ]:
mu_a = w[0].numpy()
sigma_a = tf.nn.softplus(w[1]).numpy()

mu_b = w[2].numpy()
sigma_b = tf.nn.softplus(w[3]).numpy()

print(f"a ~ N({mu_a:.3f}, {sigma_a:.3f}^2)")
print(f"b ~ N({mu_b:.3f}, {sigma_b:.3f}^2)")

In [ ]:
loss_history = logger.X[:, 9]
loss_history_kl = logger.X[:, 10]
loss_history_nll = logger.X[:, 11]

plt.plot(loss_history, label="Total")
plt.plot(loss_history_kl, label="KL")
plt.plot(loss_history_nll, label="NLL")
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Evolución de la pérdida")
plt.show()

In [ ]:
W = logger.X
weights = W[:, 1:5]

plt.plot(weights[:, 0], linestyle="-.")
plt.plot(weights[:, 2], linestyle="-.")
plt.xlabel("Epochs")
plt.ylabel("Parameter values")
plt.legend(("mu_a [VI]", "mu_b [VI]"))
plt.title("Convergencia de las medias variacionales")
plt.show()

## Predicciones con incertidumbre

Como tenemos distribuciones para $a$ y $b$, podemos muestrear muchas rectas posibles.
Esto permite visualizar incertidumbre asociada a los parámetros.

In [ ]:

# Malla de valores para x
x_grid_np = np.linspace(-3, 3, 200).reshape(-1, 1).astype("float32")
x_grid = tf.constant(x_grid_np)

# número de muestras
num_samples = 200

# truco de reparametrización para muestreo de pendientes e interceptos
a_samples = mu_a + sigma_a * np.random.randn(num_samples)
b_samples = mu_b + sigma_b * np.random.randn(num_samples)


# generación de muchas rectas
y_samples = np.array([
    a_s * x_grid_np[:, 0] + b_s
    for a_s, b_s in zip(a_samples, b_samples)
])
# tenemos 200 rectas y cada recta con 200 puntos en x
print(y.shape)


y_mean = y_samples.mean(axis=0)

y_low = np.percentile(y_samples, 5, axis=0)
y_high = np.percentile(y_samples, 95, axis=0)

plt.scatter(x_np, y_np, c="red", label="Datos")
plt.plot(x_grid_np[:, 0], y_mean, label="Media predictiva")
plt.fill_between(x_grid_np[:, 0], y_low, y_high, alpha=0.3, label="Intervalo 90%")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Predicción variacional")
plt.show()

# Preguntas:

1. ¿Qué representan la región sombreada?

2. ¿Qué significa que la banda sombreada sea ancha?

3. ¿Que incertidumbre estamos midiendo acá?

4. ¿Por qué el intervalo mostrado corresponde aproximadamente a un intervalo de confianza de 90%?

5. ¿Qué ventaja tiene este enfoque respecto a la regresión lineal tradicional usando MSE?

6. ¿El modelo estaría más seguro o inseguro al extrapolar (al predecir en un rango aidicional al de entrenamiento)?

---


## Versión tipo Bayesian Neural Network

Ahora extendemos la idea de Inferencia Variacional a una capa neuronal.

En vez de aprender únicamente dos parámetros probabilísticos $a$ y $b$, ahora aprenderemos distribuciones probabilísticas para todos los pesos y bias de una capa densa.

Cada peso tendrá una distribución variacional:

$$
q(w)=\mathcal{N}(\mu_w,\sigma_w^2)
$$

Durante entrenamiento:

* los pesos se muestrean usando el truco de reparametrización
* y se optimiza la misma función de pérdida variacional:
  * negative log-likelihood (NLL)
  * más divergencia KL.

En este ejemplo, la capa sigue siendo lineal, ya que implementa una transformación de la forma:

$$
y = XW + b
$$

Es decir, aunque ahora los pesos son probabilísticos, la relación respecto a la entrada sigue siendo lineal.

Sin embargo, esta infraestructura permite luego construir verdaderas Bayesian Neural Networks no lineales agregando:

* múltiples capas
* y funciones de activación como ReLU o tanh.

Por lo tanto, esta implementación puede verse como:

* una regresión lineal bayesiana más general
* o el bloque base para construir redes neuronales bayesianas más complejas.

In [ ]:
class BayesianDense(tf.keras.layers.Layer):
    def __init__(self, units, prior_std=1.0, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.prior_std = prior_std
        self.kl_weight = kl_weight

    def build(self, input_shape):
        input_dim = int(input_shape[-1])

        self.kernel_mu = self.add_weight(
            name="kernel_mu",
            shape=(input_dim, self.units),
            initializer="random_normal",
            trainable=True,
        )
        self.kernel_rho = self.add_weight(
            name="kernel_rho",
            shape=(input_dim, self.units),
            initializer=tf.keras.initializers.Constant(-3.0),
            trainable=True,
        )

        self.bias_mu = self.add_weight(
            name="bias_mu",
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
        )
        self.bias_rho = self.add_weight(
            name="bias_rho",
            shape=(self.units,),
            initializer=tf.keras.initializers.Constant(-3.0),
            trainable=True,
        )

    def call(self, inputs, training=True):
        kernel_sigma = tf.nn.softplus(self.kernel_rho)
        bias_sigma = tf.nn.softplus(self.bias_rho)

        if training:
            # muestreo de pesos usando ruido normal estándar
            kernel_eps = tf.random.normal(shape=tf.shape(self.kernel_mu))
            bias_eps = tf.random.normal(shape=tf.shape(self.bias_mu))

            # reparametrización
            kernel = self.kernel_mu + kernel_sigma * kernel_eps
            bias = self.bias_mu + bias_sigma * bias_eps
        else:
            # durante inferencia se usa la media --> predicción determinista
            # Para estimar incertidumbre predictiva, podríamos hacer varias pasadas 
            # con training=True o modificar la capa para muestrear también en inferencia.
            kernel = self.kernel_mu
            bias = self.bias_mu

        # KL de los pesos y los sesgos
        kernel_kl = tf.reduce_sum(kl_normal_q_p(self.kernel_mu, kernel_sigma, sigma_p=self.prior_std))
        bias_kl = tf.reduce_sum(kl_normal_q_p(self.bias_mu, bias_sigma, sigma_p=self.prior_std))

        # agregar KL a la pérdida
        # ahora tenemos kl_weight porque tenemos muchos pesos y el 
        # término KL podría crecer mucho --> estabiliza el entrenamiento
        self.add_loss(self.kl_weight * (kernel_kl + bias_kl))

        return tf.matmul(inputs, kernel) + bias

In [ ]:
class BayesianLinearRegression(tf.keras.Model):
    def __init__(self, kl_weight=1.0):
        super().__init__()
        # solo una capa con una neurona de salida
        self.linear = BayesianDense(units=1, kl_weight=kl_weight)

    def call(self, inputs, training=True):
        return self.linear(inputs, training=training)

In [ ]:
bnn = BayesianLinearRegression(kl_weight=1.0)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.005)

epochs_bnn = 5000
losses_bnn = []

for epoch in range(epochs_bnn):
    with tf.GradientTape() as tape:
        y_loc = bnn(x, training=True)

        loss_nll = -tf.reduce_sum(normal_log_prob(y, loc=y_loc, scale=sigma))
        loss_kl = tf.add_n(bnn.losses)
        total_loss = loss_nll + loss_kl

    gradients = tape.gradient(total_loss, bnn.trainable_variables)
    optimizer.apply_gradients(zip(gradients, bnn.trainable_variables))

    losses_bnn.append(total_loss.numpy())

plt.plot(losses_bnn)
plt.xlabel("Epochs")
plt.ylabel("Total loss")
plt.title("Entrenamiento BNN simple")
plt.show()

In [ ]:
layer = bnn.linear

kernel_mu = layer.kernel_mu.numpy()[0, 0]
kernel_sigma = tf.nn.softplus(layer.kernel_rho).numpy()[0, 0]

bias_mu = layer.bias_mu.numpy()[0]
bias_sigma = tf.nn.softplus(layer.bias_rho).numpy()[0]

print(f"a ~ N({kernel_mu:.3f}, {kernel_sigma:.3f}^2)")
print(f"b ~ N({bias_mu:.3f}, {bias_sigma:.3f}^2)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# parámetros aprendidos
a_mu = kernel_mu
a_sigma = kernel_sigma

b_mu = bias_mu
b_sigma = bias_sigma

# rango para graficar
a_x = np.linspace(a_mu - 4*a_sigma, a_mu + 4*a_sigma, 500)
b_x = np.linspace(b_mu - 4*b_sigma, b_mu + 4*b_sigma, 500)

# densidades gaussianas
a_pdf = norm.pdf(a_x, loc=a_mu, scale=a_sigma)
b_pdf = norm.pdf(b_x, loc=b_mu, scale=b_sigma)

# gráfico peso a
plt.figure(figsize=(7,4))

plt.plot(a_x, a_pdf)
plt.axvline(a_mu, linestyle="--", label=f"μ = {a_mu:.3f}")

plt.xlabel("a")
plt.ylabel("Densidad")

plt.title("Distribución variacional del peso a")
plt.legend()

plt.show()

# gráfico bias b
plt.figure(figsize=(7,4))

plt.plot(b_x, b_pdf)
plt.axvline(b_mu, linestyle="--", label=f"μ = {b_mu:.3f}")

plt.xlabel("b")
plt.ylabel("Densidad")

plt.title("Distribución variacional del bias b")
plt.legend()

plt.show()

In [ ]:
predictions = []

for _ in range(200):
    # con parámetro training = True, estoy prediciendo
    # pero sampleando pesos aleatorios
    # con training = False, usa las medias --> determinista
    y_pred = bnn(x_grid, training=True)
    predictions.append(y_pred.numpy())

predictions = np.array(predictions)

y_mean = predictions.mean(axis=0)
y_low = np.percentile(predictions, 5, axis=0)
y_high = np.percentile(predictions, 95, axis=0)

In [ ]:
x_grid_np = np.linspace(-3, 3, 200).reshape(-1, 1).astype("float32")

num_samples = 200
predictions = []

for _ in range(num_samples):
    y_pred = bnn(x_grid_np, training=True)
    predictions.append(y_pred.numpy().squeeze())

predictions = np.array(predictions)

y_mean = predictions.mean(axis=0).squeeze()
y_low = np.percentile(predictions, 5, axis=0).squeeze()
y_high = np.percentile(predictions, 95, axis=0).squeeze()

x_plot = x_grid_np[:, 0]

plt.figure(figsize=(8,5))

plt.scatter(x_np.squeeze(), y_np.squeeze(), c="red", label="Datos")

plt.plot(
    x_plot,
    y_mean,
    linewidth=2,
    label="Media predictiva"
)

plt.fill_between(
    x_plot,
    y_low,
    y_high,
    alpha=0.3,
    label="Intervalo 90%"
)

plt.xlabel("x")
plt.ylabel("y")
plt.title("Predicción Bayesiana")
plt.legend()
plt.show()


# Preguntas

1. En el modelo:

$$
\texttt{self.linear = BayesianDense(units=1, kl\_weight=kl\_weight)}
$$

¿por qué el modelo sigue siendo lineal, aunque ahora los pesos sean probabilísticos?



2. ¿Qué diferencia conceptual existe entre:

* una regresión lineal clásica
* y esta versión bayesiana de regresión lineal?



3. En la capa `BayesianDense`, ¿por qué no se aprende directamente $\sigma$, sino un parámetro $\rho$ tal que:

$$
\sigma = \text{softplus}(\rho)
$$



4. Explique qué ocurre durante esta parte del código:

```python
kernel = self.kernel_mu + kernel_sigma * kernel_eps
```

5. ¿Qué modificaciones habría que hacer al modelo actual para que pudiera aprender funciones no lineales más complejas?

---